# 🚀 PeakInfer - GPU/TPU Detection Test

This notebook tests PeakInfer's hardware detection capabilities on Google Colab.

**Instructions:**
1. Go to **Runtime → Change runtime type**
2. Select **GPU** (T4) or **TPU** (v2)
3. Run all cells

---

## 1️⃣ Check Current Hardware

In [ ]:
# Check GPU
!nvidia-smi

# Check CUDA
!nvcc --version 2>/dev/null || echo 'CUDA not found'

# Environment variables
import os
print(f"\nCUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES', 'not set')}")

In [ ]:
# Check TPU (if TPU runtime selected)
import os

try:
    if 'COLAB_TPU_ADDR' in os.environ:
        print(f"✅ TPU available at: {os.environ['COLAB_TPU_ADDR']}")
        
        # JAX TPU detection
        import jax
        print(f"JAX devices: {jax.devices()}")
    else:
        print("⚠️ No TPU configured. Go to Runtime → Change runtime type → TPU")
except Exception as e:
    print(f"TPU check: {e}")

## 2️⃣ Install PeakInfer

In [ ]:
# Install Node.js (required for PeakInfer CLI)
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs
!node --version
!npm --version

In [ ]:
# Clone and install PeakInfer
!git clone https://github.com/kalmantic/peakinfer.git 2>/dev/null || (cd peakinfer && git pull)
%cd peakinfer
!npm install
!npm run build

## 3️⃣ Create Sample Inference Code

Creating sample files with GPU/TPU patterns for PeakInfer to detect.

In [ ]:
%%writefile /content/sample_inference.py
"""
Sample LLM Inference Code for PeakInfer Detection
"""

# =============================================================================
# GPU Detection Patterns
# =============================================================================

import torch

# Check GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_count = torch.cuda.device_count()
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu_name}, Count: {gpu_count}")

# =============================================================================
# vLLM Configuration
# =============================================================================

from vllm import LLM, SamplingParams

llm = LLM(
    model="meta-llama/Llama-3.1-8B-Instruct",
    tensor_parallel_size=1,
    gpu_memory_utilization=0.9,
    enable_prefix_caching=True,
    max_model_len=4096,
)

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=512,
)

# =============================================================================
# LLM API Calls (Application Layer)
# =============================================================================

from openai import OpenAI
client = OpenAI()

def generate_response(prompt: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100,
        temperature=0,  # Cacheable!
    )
    return response.choices[0].message.content

# =============================================================================
# Quantization (4-bit)
# =============================================================================

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
%%writefile /content/tpu_inference.py
"""
TPU Inference Patterns for PeakInfer Detection
"""

import jax
import jax.numpy as jnp

# TPU device detection
devices = jax.devices('tpu')
print(f"TPU devices: {devices}")

# XLA TPU configuration
import os
os.environ['XRT_TPU_CONFIG'] = 'localservice;0;localhost:51011'
os.environ['TPU_NAME'] = 'local'

# Flax/JAX model on TPU
from flax import linen as nn

class TransformerBlock(nn.Module):
    features: int = 768
    num_heads: int = 12
    
    @nn.compact
    def __call__(self, x):
        attention = nn.MultiHeadDotProductAttention(num_heads=self.num_heads)
        x = attention(x, x)
        return x

# TPU mesh for distributed training
from jax.experimental import mesh_utils
from jax.sharding import PositionalSharding

devices = mesh_utils.create_device_mesh((8,))
sharding = PositionalSharding(devices)

## 4️⃣ Run PeakInfer Hardware Detection

In [ ]:
# Run PeakInfer hardware detection on /content directory
!node -e "
import('./dist/collectors/hardware-detector.js').then(async ({ HardwareDetector }) => {
  const detector = new HardwareDetector('/content', true);
  const profile = await detector.detect();
  
  console.log('\n' + '='.repeat(60));
  console.log('🖥️  PEAKINFER HARDWARE DETECTION RESULTS');
  console.log('='.repeat(60));
  
  console.log('\n📊 Summary:');
  console.log(JSON.stringify(profile.summary, null, 2));
  
  if (profile.gpus.length > 0) {
    console.log('\n🎮 GPUs Detected:');
    profile.gpus.forEach(gpu => console.log('  -', gpu.type, 'x' + gpu.count));
  }
  
  if (profile.tpus.length > 0) {
    console.log('\n⚡ TPUs Detected:');
    profile.tpus.forEach(tpu => console.log('  -', tpu.type, 'source:', tpu.source));
  }
  
  if (profile.servingRuntimes.length > 0) {
    console.log('\n🚀 Serving Runtimes:');
    profile.servingRuntimes.forEach(rt => console.log('  -', rt.runtime));
  }
  
  if (profile.quantization.length > 0) {
    console.log('\n📦 Quantization:');
    profile.quantization.forEach(q => console.log('  -', q.method, q.bits + '-bit'));
  }
  
  if (profile.parallelization.length > 0) {
    console.log('\n⚡ Parallelization:');
    profile.parallelization.forEach(p => console.log('  -', p.strategy));
  }
  
  console.log('\n' + '='.repeat(60));
}).catch(console.error);
"

## 5️⃣ Run Full Codebase Analysis

In [ ]:
# Run full codebase analysis
!node -e "
import('./dist/collectors/codebase-collector.js').then(async ({ CodebaseCollector }) => {
  const collector = new CodebaseCollector({
    rootPath: '/content',
    scanDepth: 'deep',
  });
  
  const analysis = await collector.analyzeCodebase();
  
  console.log('\n' + '='.repeat(60));
  console.log('📊 PEAKINFER CODEBASE ANALYSIS');
  console.log('='.repeat(60));
  
  console.log('\n🔍 LLM API Calls Found:', analysis.llmApiCalls.length);
  analysis.llmApiCalls.slice(0, 5).forEach(call => {
    console.log('  -', call.provider + '/' + call.model, 'in', call.location);
  });
  
  console.log('\n🎯 Optimization Opportunities:', analysis.optimizationOpportunities.length);
  analysis.optimizationOpportunities.slice(0, 5).forEach(opt => {
    console.log('  - [' + opt.priority + ']', opt.type + ':', opt.description.slice(0, 50) + '...');
  });
  
  if (analysis.hardwareProfile) {
    console.log('\n🖥️  Hardware Profile:');
    console.log('  - GPUs:', analysis.hardwareProfile.summary.totalGPUs);
    console.log('  - Runtimes:', analysis.hardwareProfile.servingRuntimes.map(r => r.runtime).join(', '));
  }
  
  console.log('\n' + '='.repeat(60));
}).catch(console.error);
"

## 6️⃣ Real GPU Info (PyTorch)

In [ ]:
import torch

print("=" * 60)
print("🎮 PYTORCH GPU DETECTION")
print("=" * 60)

if torch.cuda.is_available():
    print(f"\n✅ CUDA Available: {torch.cuda.is_available()}")
    print(f"📊 GPU Count: {torch.cuda.device_count()}")
    
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"\n🖥️  GPU {i}: {props.name}")
        print(f"   Memory: {props.total_memory / 1024**3:.1f} GB")
        print(f"   Compute Capability: {props.major}.{props.minor}")
        print(f"   Multi Processors: {props.multi_processor_count}")
else:
    print("\n⚠️ No GPU available. Go to Runtime → Change runtime type → GPU")

print("\n" + "=" * 60)

## 📋 Full-Stack Inference View

This is what PeakInfer provides - a unified view across all layers:

```
┌─────────────────────────────────────────┐
│  APPLICATION LAYER                      │
│    • LLM API calls (OpenAI, Anthropic)  │
│    • Caching opportunities              │
│    • Model routing suggestions          │
├─────────────────────────────────────────┤
│  SERVING LAYER                          │
│    • vLLM, TensorRT-LLM, SGLang         │
│    • Parallelization strategies         │
│    • Quantization methods               │
├─────────────────────────────────────────┤
│  INFRASTRUCTURE LAYER                   │
│    • GPU/TPU detection                  │
│    • Cloud instance optimization        │
│    • Cost reduction recommendations     │
└─────────────────────────────────────────┘
```